# Customer Intelligence & Growth Project

## Objetivo
Projeto completo de Customer Intelligence focado em:
- **Churn Prediction**: identificar clientes com risco de cancelamento
- **Propensity Modeling**: prever probabilidade de compra/renovação
- **Recommendation**: sugerir próxima melhor ação
- **Segmentation**: agrupar clientes por comportamento
- **A/B Testing**: medir efetividade de campanhas
- **Causal Inference**: entender impacto causal de ações

## Arquitetura
```
Bronze → Silver → Gold → Models → Scoring
   ↓        ↓       ↓        ↓        ↓
 Raw    Clean   Features  ML     Predictions
```

## Estrutura de Pastas
- `00_setup/`: Configuração inicial
- `01_bronze/`: Ingestão de dados raw
- `02_silver/`: Limpeza e transformação
- `03_gold/`: Feature engineering
- `04_models/`: Treinamento de modelos
- `05_scoring/`: Batch scoring
- `06_experimentation/`: A/B testing e causalidade
- `07_monitoring/`: Drift e performance
- `08_dashboards/`: SQL queries para visualização

In [ ]:
# Configurações globais do projeto
import os
from datetime import datetime

# Configurações de catálogo e schema
CATALOG = "customer_intelligence"  # Catálogo dedicado ao projeto
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"
SCHEMA_GOLD = "gold"

# Configurações MLflow (usuário detectado dinamicamente, funciona em qualquer conta)
CURRENT_USER = spark.sql("SELECT current_user()").collect()[0][0]
MLFLOW_EXPERIMENT_PATH = f"/Users/{CURRENT_USER}/customer_intelligence_experiments"

# Configurações gerais
DATA_PATH = "/FileStore/customer_intelligence/data"
MODEL_REGISTRY_NAME_PREFIX = "customer_intelligence"

# Armazenar em widgets para reutilização
dbutils.widgets.text("catalog", CATALOG, "Catalog")
dbutils.widgets.text("schema_bronze", SCHEMA_BRONZE, "Bronze Schema")
dbutils.widgets.text("schema_silver", SCHEMA_SILVER, "Silver Schema")
dbutils.widgets.text("schema_gold", SCHEMA_GOLD, "Gold Schema")

print(f"✓ Configurações carregadas")
print(f"  Catalog: {CATALOG}")
print(f"  Bronze: {SCHEMA_BRONZE}")
print(f"  Silver: {SCHEMA_SILVER}")
print(f"  Gold: {SCHEMA_GOLD}")
print(f"  Usuário: {CURRENT_USER}")

In [ ]:
%sql
-- Criar o catálogo do projeto (necessário em qualquer workspace novo)
CREATE CATALOG IF NOT EXISTS customer_intelligence
  COMMENT 'Catálogo dedicado ao projeto Customer Intelligence';

-- Criar schemas para o projeto
CREATE SCHEMA IF NOT EXISTS customer_intelligence.bronze
  COMMENT 'Raw data - camada bronze para ingestão inicial';

CREATE SCHEMA IF NOT EXISTS customer_intelligence.silver
  COMMENT 'Clean data - camada silver com dados limpos e transformados';

CREATE SCHEMA IF NOT EXISTS customer_intelligence.gold
  COMMENT 'Feature store - camada gold com features agregadas e scores';

SHOW SCHEMAS IN customer_intelligence;

In [0]:
# Funções auxiliares para o projeto

def get_full_table_name(schema, table):
    """Retorna nome completo da tabela"""
    return f"{CATALOG}.{schema}.{table}"

def create_or_replace_table(df, schema, table, partition_by=None):
    """Salva DataFrame como tabela Delta"""
    full_name = get_full_table_name(schema, table)
    writer = df.write.format("delta").mode("overwrite")
    if partition_by:
        writer = writer.partitionBy(partition_by)
    writer.saveAsTable(full_name)
    print(f"✓ Tabela criada: {full_name}")
    return full_name

def get_latest_model_version(model_name):
    """Retorna a última versão de um modelo no MLflow"""
    from mlflow.tracking import MlflowClient
    client = MlflowClient()
    try:
        versions = client.search_model_versions(f"name='{model_name}'")
        if versions:
            latest = max([int(v.version) for v in versions])
            return latest
    except:
        pass
    return None

def log_metrics_to_mlflow(metrics_dict, step=None):
    """Log múltiplas métricas no MLflow"""
    import mlflow
    for key, value in metrics_dict.items():
        mlflow.log_metric(key, value, step=step)

print("✓ Helper functions carregadas")

In [0]:
# Verificar se tudo está configurado
import mlflow

print("="*60)
print("VERIFICAÇÃO DE SETUP")
print("="*60)

# 1. Verificar schemas
try:
    schemas = spark.sql(f"SHOW SCHEMAS IN {CATALOG}").collect()
    print(f"\n✓ Schemas criados: {len(schemas)}")
    for schema in schemas:
        print(f"  - {schema.databaseName}")
except Exception as e:
    print(f"✗ Erro ao verificar schemas: {e}")

# 2. Verificar MLflow
try:
    mlflow.set_experiment(MLFLOW_EXPERIMENT_PATH)
    print(f"\n✓ MLflow experiment configurado: {MLFLOW_EXPERIMENT_PATH}")
except Exception as e:
    print(f"✗ Erro ao configurar MLflow: {e}")

# 3. Resumo
print("\n" + "="*60)
print("SETUP COMPLETO ✓")
print("="*60)
print("\nPróximos passos:")
print("1. Execute notebooks em 01_bronze/ para ingerir dados")
print("2. Execute notebooks em 02_silver/ para limpar dados")
print("3. Execute notebooks em 03_gold/ para criar features")
print("4. Execute notebooks em 04_models/ para treinar modelos")
print("5. Execute notebooks em 05_scoring/ para gerar previsões")